# Email Finder — Part 2: Fetch Mailin Results + Export

Run this notebook **after Mailin shows status = Completed** in the Task Results tab.

Pipeline:
1. Load task state saved by `test_email_finder.ipynb`
2. Fetch completed results from Mailin
3. Apply verification statuses to discovery results
4. Preview final results
5. Export enriched + still_needs CSVs

## Cell 1: Setup & Config

In [ ]:
import sys, json, logging
sys.path.append("../..")

from email_finder import LeadInput, EmailFinderResult, export_results
from email_finder.config import Config
from email_finder.batch import apply_mailin_results
from email_finder.verification.mailin_automator import mailin_fetch_results

logging.basicConfig(level=logging.INFO)
config = Config()
print("Config loaded.")

## Cell 2: Load Task State

In [ ]:
STATE_PATH = "./output/mailin_task_state.json"

with open(STATE_PATH) as f:
    state = json.load(f)

task_id       = state["task_id"]
all_candidates = state["all_candidates"]
leads         = [LeadInput(**d) for d in state["leads"]]
pre_results   = [
    EmailFinderResult(
        email=d["email"],
        status=d["status"],
        confidence=d["confidence"],
        source=d["source"],
        discovery_log=d["discovery_log"],
        verification_details=d["verification_details"],
    )
    for d in state["pre_results"]
]

print(f"Task ID        : {task_id}")
print(f"Submitted at   : {state['submitted_at']}")
print(f"Leads          : {len(leads)}")
print(f"Candidates     : {len(all_candidates)}")

## Cell 3: Fetch Results from Mailin

Opens a browser, navigates to Task Results, and downloads the CSV.  
Raises `RuntimeError` if the task is still **Verifying** — wait and retry.

In [ ]:
verification_map = await mailin_fetch_results(task_id, all_candidates, config, headless=True)

# Summary
from collections import Counter
counts = Counter(verification_map.values())
print(f"Verification results for {len(verification_map)} emails:")
for status, n in sorted(counts.items()):
    print(f"  {status:<12} {n}")

## Cell 4: Apply Results + Preview

In [ ]:
results = apply_mailin_results(leads, pre_results, verification_map)

status_icon = {"verified": "✓", "catch_all": "~", "unverified": "?", "not_found": "✗", "invalid": "✗"}

print(f"{'Name':<35}  {'Email':<40}  {'Status':<14}  Conf")
print("-" * 108)
for lead, result in zip(leads, results):
    icon = status_icon.get(result.status, "?")
    email_str = result.email or "N/A"
    conf = f"{result.confidence:.0%}" if result.confidence else ""
    print(f"[{icon}] {lead.full_name:<33}  {email_str:<40}  {result.status:<14}  {conf}")

## Cell 5: Debug — Deep Dive on One Lead

Change `idx` to inspect a different lead.

In [ ]:
idx = 0
lead = leads[idx]
result = results[idx]

print(f"Lead:       {lead.full_name}")
print(f"Email:      {result.email}")
print(f"Status:     {result.status}")
print(f"Confidence: {result.confidence:.0%}")
print(f"Source:     {result.source}")
print()
print("Discovery log:")
for entry in result.discovery_log:
    node = entry.get('node', '?')
    res  = entry.get('result', {})
    found = res.get('found_email') or res.get('found_emails') or '—'
    err   = res.get('error') or ''
    print(f"  [{node:<20}]  found={found}  {'ERROR: ' + err if err else ''}")

## Cell 6: Export to CSV

In [ ]:
files = export_results(leads, results, output_dir="./output", prefix="enriched")
print(f"\nEnriched:      {files['enriched']}")
print(f"Still needs:   {files['needs_enrichment']}")